# 06 — Bölgeler Arası Genelleme (AS5)

**Soru:** Bir grup şehirde eğitilen sınıflandırma modeli, hiç görmediği bir şehirde ne kadar başarılı?

**Yöntem: Leave-One-City-Out (LOCO, "bir şehri dışarıda bırak").** 7 tur yapılıyor. Her turda bir şehir tamamen dışarıda bırakılıyor; model kalan 6 şehrin **tüm** noktalarıyla eğitilip dışarıda bırakılan şehirde test ediliyor.

**Karşılaştırma ("aynı şehir" ve "başka şehir"):**
- **Aynı şehir (Faz 3 düzeni):** Model 7 şehrin eğitim noktalarıyla eğitildi. Test edilen şehrin başka noktalarını eğitimde görmüştü.
- **Başka şehir (LOCO):** Test edilen şehirden hiç nokta görülmedi.

Adil bir karşılaştırma için iki düzen de **aynı test noktalarında** (her şehrin Faz 3 test noktaları) değerlendiriliyor. Aradaki fark **genelleme kaybıdır.**

Hiperparametreler Faz 3'te seçilen değerler; LOCO'da yeniden arama yapılmıyor. Böylece fark yalnızca "şehri görmüş olmak" etkisini gösteriyor. Earth Engine kullanılmıyor.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) özel GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Özel repoyu okumak için Colab'ın **Gizli anahtarlar (Secrets)** bölümündeki `GITHUB_TOKEN` kullanılır. Token ekrana yazdırılmaz.

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + (r.stderr or r.stdout).replace(token, "***"))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Colab'da sol menüdeki anahtar simgesinden GITHUB_TOKEN ekleyin ve "
                           "'Not defteri erişimi'ni açın (README'ye bakın).") from e
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import ml_utils as mu
importlib.reload(mu)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F, M = config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

df = pd.read_csv(f"{config.DATA_PROCESSED}/samples_2021.csv").merge(
    pd.read_csv(f"{config.DATA_PROCESSED}/split_assignments.csv")[["sample_id", "split"]], on="sample_id")
meta = json.load(open(f"{M}/best_model.json", encoding="utf-8"))
FEATS = config.MODEL_FEATURES
LABELS = sorted(config.CLASSES)
CLASS_NAMES = [config.CLASSES[k] for k in LABELS]
CITY_LIST = list(config.CITIES)
MODELS = config.MODELS
BEST = meta["best_model"]
print("En iyi model (Faz 3):", BEST, "| tüm modeller LOCO'da değerlendiriliyor:", MODELS)

## 1. Aynı şehir: Faz 3 modelleri, şehir bazında test sonuçları

In [ ]:
same_rows, same_pred = [], {}
for name in MODELS:
    model = joblib.load(f"{M}/{name}.joblib")["model"]
    for city in CITY_LIST:
        t = df[(df["city"] == city) & (df["split"] == "test")]
        p, pr = model.predict(t[FEATS]), model.predict_proba(t[FEATS])
        same_pred[(name, city)] = p
        same_rows.append({"model": name, "test_city": city, "duzen": "Aynı şehir",
                          "n_train": int((df["split"] == "train").sum()), "n_test": len(t),
                          **mu.compute_metrics(t["class"], p, pr, LABELS)})
same = pd.DataFrame(same_rows)
same[same["model"] == BEST][["test_city", "accuracy", "f1_macro", "roc_auc_ovr_macro", "kappa"]].round(4)

## 2. Başka şehir: Leave-One-City-Out
Her turda model sıfırdan eğitiliyor (4 model × 7 şehir = 28 eğitim). SVM'in olasılık hesabı en yavaş adım.

Ayrıca dışarıda bırakılan şehrin **tüm** noktalarındaki başarı da raporlanıyor (`n_test` daha büyük, sonuç daha kararlı).

In [ ]:
loco_csv = f"{T}/genelleme_ham.csv"
rows = []
for city in CITY_LIST:
    tr = df[df["city"] != city]
    for name in MODELS:
        model = mu.make_pipeline(name, meta["all_params"][name]).fit(tr[FEATS], tr["class"])
        for scope, t in [("Başka şehir (LOCO)", df[(df["city"] == city) & (df["split"] == "test")]),
                         ("Başka şehir (LOCO, tüm noktalar)", df[df["city"] == city])]:
            p, pr = model.predict(t[FEATS]), model.predict_proba(t[FEATS])
            rows.append({"model": name, "test_city": city, "duzen": scope, "n_train": len(tr), "n_test": len(t),
                         **mu.compute_metrics(t["class"], p, pr, LABELS)})
            if scope == "Başka şehir (LOCO)" and name == BEST:
                same_pred[("LOCO", city)] = p
    print(f"{city} tamam")

res = pd.concat([same, pd.DataFrame(rows)], ignore_index=True).round(4)
res.to_csv(loco_csv, index=False, encoding="utf-8-sig")
gen = res[["model", "test_city", "duzen", "n_train", "n_test", "accuracy", "f1_macro", "iou_macro", "roc_auc_ovr_macro", "kappa"]]
gen.to_csv(f"{T}/genelleme.csv", index=False, encoding="utf-8-sig")
gen[gen["model"] == BEST]

## 3. Şehir × metrik ısı haritası ve performans düşüşü
**Düşüş** = Aynı şehir − Başka şehir (aynı test noktalarında). Pozitif değer, şehri görmemenin başarıyı ne kadar azalttığını gösterir.

In [ ]:
METRICS = {"accuracy": "Accuracy", "f1_macro": "macro-F1", "roc_auc_ovr_macro": "ROC-AUC"}
b = gen[gen["model"] == BEST]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, scope in zip(axes, ["Aynı şehir", "Başka şehir (LOCO)"]):
    piv = b[b["duzen"] == scope].set_index("test_city").loc[CITY_LIST, list(METRICS)].rename(columns=METRICS)
    sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlGnBu", vmin=gen[list(METRICS)].min().min(), vmax=1, ax=ax, cbar=ax is axes[1])
    ax.set_title(f"{BEST} — {scope}"); ax.set_ylabel("Test şehri")
savefig("genelleme_isi_haritasi")

same_ = gen[gen["duzen"] == "Aynı şehir"].set_index(["model", "test_city"])
loco_ = gen[gen["duzen"] == "Başka şehir (LOCO)"].set_index(["model", "test_city"])
drop = (same_[list(METRICS)] - loco_[list(METRICS)]).add_prefix("dusus_").reset_index().round(4)
drop.to_csv(f"{T}/genelleme_dusus.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
piv = drop.pivot(index="test_city", columns="model", values="dusus_f1_macro").loc[CITY_LIST, MODELS]
piv.plot(kind="bar", ax=axes[0], width=0.8, colormap="Set2")
axes[0].axhline(0, color="k", lw=0.8); axes[0].set_ylabel("macro-F1 düşüşü (aynı − başka şehir)")
axes[0].set_title("Şehri görmemenin maliyeti (model bazında)"); axes[0].tick_params(axis="x", rotation=30)
mean_tab = gen[gen["duzen"].isin(["Aynı şehir", "Başka şehir (LOCO)"])].groupby(["model", "duzen"])["f1_macro"].mean().unstack()
mean_tab[["Aynı şehir", "Başka şehir (LOCO)"]].plot(kind="bar", ax=axes[1], color=["#4575b4", "#d73027"], width=0.7)
axes[1].set_ylabel("Ortalama macro-F1 (7 şehir)"); axes[1].set_title("Aynı şehir vs başka şehir")
axes[1].set_ylim(max(0, mean_tab.values.min() - 0.15), 1); axes[1].tick_params(axis="x", rotation=0)
savefig("genelleme_dusus")
mean_tab.assign(fark=lambda d: d["Aynı şehir"] - d["Başka şehir (LOCO)"]).round(4)

## 4. Hangi sınıflar genellenemiyor?
En iyi model için, her şehirde LOCO düzeninde sınıf bazında F1 ve aynı şehir düzenine göre düşüş.

In [ ]:
from sklearn.metrics import f1_score
rows = []
for city in CITY_LIST:
    t = df[(df["city"] == city) & (df["split"] == "test")]
    f_same = f1_score(t["class"], same_pred[(BEST, city)], labels=LABELS, average=None, zero_division=0)
    f_loco = f1_score(t["class"], same_pred[("LOCO", city)], labels=LABELS, average=None, zero_division=0)
    for k, a, l in zip(LABELS, f_same, f_loco):
        rows.append({"test_city": city, "sinif": config.CLASSES[k], "f1_ayni": a, "f1_loco": l, "dusus": a - l})
cls_drop = pd.DataFrame(rows).round(4)
cls_drop.to_csv(f"{T}/genelleme_sinif_bazinda.csv", index=False, encoding="utf-8-sig")
plt.figure(figsize=(9, 5))
sns.heatmap(cls_drop.pivot(index="test_city", columns="sinif", values="dusus").loc[CITY_LIST, CLASS_NAMES],
            annot=True, fmt="+.2f", cmap="Reds", center=0, cbar_kws={"label": "F1 düşüşü"})
plt.title(f"{BEST} — sınıf bazında F1 düşüşü (aynı şehir − LOCO)"); plt.ylabel("")
savefig("genelleme_sinif_bazinda")

## 5. Düşüşün olası nedenleri: şehirler ne kadar farklı?
Genelleme kaybını açıklayabilecek iki ölçü:
- **Spektral kayma:** Dışarıda bırakılan şehrin özellik değerleri, aynı sınıf için diğer 6 şehirden ne kadar farklı? Sınıf başına standartlaştırılmış ortalama farkı (|ortalama farkı| / std) hesaplanıyor, sonra sınıflar ve özellikler üzerinden ortalaması alınıyor. Büyükse, o şehirde aynı sınıf "başka türlü görünüyor" demektir.
- **İklim farkı:** Şehrin 2003–2024 ortalama yaz LST'si, yağışı ve toprak nemi (Faz 1 iklim serisi).

Aşağıdaki tablo bu ölçüleri F1 düşüşüyle yan yana koyuyor. **Uyarı:** 7 şehirle hesaplanan korelasyon çok kırılgandır. Kanıt değil, yalnızca yorumu destekleyen bir gösterge olarak kullanın.

In [ ]:
rows = []
for city in CITY_LIST:
    smd = []
    for k in LABELS:
        a = df[(df["city"] == city) & (df["class"] == k)][FEATS]
        o = df[(df["city"] != city) & (df["class"] == k)][FEATS]
        if len(a) < 5 or len(o) < 5:
            continue
        smd.append(((a.mean() - o.mean()).abs() / o.std().replace(0, np.nan)).mean())
    rows.append({"test_city": city, "spektral_kayma": np.nanmean(smd)})
shift = pd.DataFrame(rows).set_index("test_city")
clim = pd.read_csv(f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv").groupby("city")[
    ["modis_lst_day", "chirps_precip_annual", "era5_sm_summer"]].mean().add_suffix("_ort")
dist = pd.crosstab(df["city"], df["class"], normalize="index")
shift["sinif_dagilimi_farki"] = [0.5 * (dist.loc[c] - dist.drop(index=c).mean()).abs().sum() for c in shift.index]
d_best = drop[drop["model"] == BEST].set_index("test_city")["dusus_f1_macro"]
why = shift.join(clim).join(d_best.rename("f1_dususu")).loc[CITY_LIST].round(4)
why.to_csv(f"{T}/genelleme_neden_gostergeleri.csv", encoding="utf-8-sig")
display(why)
why_sp = why.corr(method="spearman")["f1_dususu"].drop("f1_dususu").round(3).rename("spearman_rho_f1_dususu")
why_sp.to_csv(f"{T}/genelleme_neden_spearman.csv", encoding="utf-8-sig")
print("Spearman (7 şehir; yalnızca gösterge):")
print(why_sp)

### Düşüşün olası nedenleri (rapor için madde listesi — sayılarla kontrol edin)
- **İklim ve fenoloji farkı:** Aynı sınıf farklı iklimde farklı görünür. Örneğin Karadeniz'in nemli ormanı (Artvin, Samsun) ile Güneydoğu'nun seyrek ağaçlığı (Şanlıurfa) ya da sulu ve kuru tarım. Yaz kompozitinde kuru iklimdeki tarım alanları hasat sonrası çıplak toprağa benzer.
- **Topografya:** Dağlık illerde (Artvin, Van) gölge ve eğim yansımayı değiştirir. Modele rakım ya da eğim verilmedi.
- **Sınıf dağılımı ve nadir sınıflar:** Bir sınıf bir ilde çok nadirse (bkz. `ornek_sayilari.csv`), o şehirden hem az örnek gelir hem de test edildiğinde F1 dalgalanır.
- **Kentsel doku farkı:** İstanbul'un yoğun beton dokusu ile küçük il merkezlerinin karışık dokusu farklıdır. Yapılaşma ↔ Çıplak karışması şehirden şehre değişir.
- **Sulak alan tanımı:** Otsu sulak alanların payı illere göre değişiyor (Faz 2, 5b). Bu da Su/Sulak sınıfının başarısını şehre bağımlı yapıyor.
- **Etiket kalitesi:** WorldCover'ın doğruluğu bölgeye göre değişebilir. Bir şehirdeki "düşüş"ün bir kısmı etiket hatası olabilir.

**AS5'in cevabı için:** Ortalama macro-F1 düşüşünü (yukarıdaki tablo) ve en çok düşen şehir/sınıfları verin. "Model genellenebilir mi?" sorusuna "kısmen: şu sınıflarda evet, şu şehirlerde hayır" gibi sayıya dayalı bir cevap yazın.

## 6. H6 hipotezi ve tüm hipotezlerin özeti
**H6:** Farklı bölgelerde eğitilen modellerin başarısı değişir ve bölgesel farklılıklar genellenebilirliği etkiler.

Kanıt: her model için aynı şehir → başka şehir macro-F1 düşüşünün ortalaması ve iller arası aralığı; en iyi model için en çok ve en az düşen il; düşüşle iklim göstergeleri arasındaki Spearman (n = 7, yalnızca örüntü). Faz 4'teki H1–H5 tablosu varsa bununla birleştirilip `hipotez_ozet.csv` olarak kaydedilir. **Karar sütununu ekip yazar.**

In [ ]:
h6 = []
for name in MODELS:
    d = drop[drop["model"] == name].set_index("test_city")["dusus_f1_macro"]
    h6.append({"hipotez": "H6", "ifade": config.HYPOTHESES["H6"][0], "kanit": f"{name}: macro-F1 düşüşü (aynı → başka şehir)",
               "beklenen_yon": "düşüş > 0, iller arası fark büyük", "n_il": len(d),
               "il_rho_en_az_0.1": "—", "beklenen_yonde_il (|rho|>=0.1)": int((d > 0).sum()),
               "rho_min": d.min(), "rho_max": d.max(),
               "en_guclu_il": f"ortalama {d.mean():+.3f}; en çok {d.idxmax()} ({d.max():+.3f}), en az {d.idxmin()} ({d.min():+.3f})",
               "karar_[EKİP_YAZACAK]": ""})
for ind, rho in why_sp.items():
    h6.append({"hipotez": "H6", "ifade": config.HYPOTHESES["H6"][0], "kanit": f"F1 düşüşü – {ind} (Spearman, n = 7)",
               "beklenen_yon": "—", "n_il": 7, "il_rho_en_az_0.1": "—", "beklenen_yonde_il (|rho|>=0.1)": "—",
               "rho_min": rho, "rho_max": rho, "en_guclu_il": "yalnızca örüntü, kanıt değil", "karar_[EKİP_YAZACAK]": ""})
h6 = pd.DataFrame(h6)
h15_path = f"{T}/hipotez_ozet_H1_H5.csv"
if os.path.exists(h15_path):
    allh = pd.concat([pd.read_csv(h15_path), h6], ignore_index=True)
else:
    print("Uyarı: hipotez_ozet_H1_H5.csv yok (önce 04'ün B9 bölümünü çalıştırın). Yalnızca H6 kaydediliyor.")
    allh = h6
allh = allh.round(3)
allh.to_csv(f"{T}/hipotez_ozet.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
allh

## Faz 6 bitti mi?
- [ ] `results/tables/genelleme.csv` ve `genelleme_isi_haritasi.png`
- [ ] `genelleme_dusus.csv`, `genelleme_sinif_bazinda.csv`, `genelleme_neden_gostergeleri.csv`, `genelleme_neden_spearman.csv`
- [ ] `hipotez_ozet.csv`: H1–H6 tek tabloda (karar sütununu ekip doldurur)

## Bu fazda ne yaptık, neden yaptık
- Modelin hiç görmediği bir şehirde başarısını ölçmek için Leave-One-City-Out uyguladık. Bu, modeli yeni bir bölgeye uygulamanın gerçekçi bir simülasyonudur.
- "Aynı şehir" ve "başka şehir" sonuçlarını **aynı test noktalarında** karşılaştırdık. Böylece fark yalnızca şehri görmüş olmaktan kaynaklanıyor.
- Hiperparametreleri sabit tuttuk (Faz 3), yani düşüş ayar farkından değil, veri farkından geliyor.
- Düşüşü sınıf bazında inceledik ve spektral kayma, iklim ve sınıf dağılımı farkı gibi olası nedenleri sayısal göstergelerle destekledik.